In [ ]:
# Final Project by Corey Jordan-Zamora

# Load the data from my S3 bucket
df = spark.read.option("header", "true").option("inferSchema", "true").csv("s3://corey-final-project-bucket/")

# Show the first 5 rows to make sure it has loaded alright
#df.show(5, truncate=False)

# Drop rows with any null values
from pyspark.sql.functions import col
df_clean = df.dropna()

# Ensure proper review score of only -1 or 1
df_clean = df_clean.filter(col("review_score").isin(-1, 1))

# Now show the cleaned data to make sure it is still loading alright
#df_clean.show(5, truncate=False)

# Save as Parquet for Athena querying
df_clean.write.mode("overwrite").parquet("s3://corey-final-project-bucket/cleaned_steam_reviews/")

# Checking variable impact

# Step 1: Load Cleaned Data from Parquet
df = spark.read.parquet("s3://corey-final-project-bucket/cleaned_steam_reviews/")
#df.show(5, truncate=False)

# Step 2: Feature Engineering: Add Review Length Column thru character length route
from pyspark.sql.functions import length, size, split
# Add character count column
df = df.withColumn("review_length_chars", length("review_text"))
df.select("review_text", "review_length_chars", "review_score").show(5, truncate=False)

# Cast review_score to integer to properly deal with data (making sure it is int)
from pyspark.sql.functions import col
df = df.withColumn("review_score", col("review_score").cast("int"))
# Then drop nulls and filter
df = df.dropna(subset=["review_score", "review_text"]) # Explicitly drop nulls in relevant columns
df = df.filter(col("review_score").isin(-1, 1))

# Step 3: Basic Data Exploration
from pyspark.sql.functions import floor
# Bucket reviews into length categories
df = df.withColumn("length_bucket", floor(df["review_length_chars"] / 50) * 50)
# Average review score per bucket
df.groupBy("length_bucket").avg("review_score").orderBy("length_bucket").show()

# Step 4: Filter for Valid Scores and Explore Distribution
# Filter out rows where review_score is null or not -1 or 1
df = df.filter(df.review_score.isin(-1, 1))

# This is optional: Just viewing distribution of positive vs. negative reviews
df.groupBy("review_score").count().show()

# Step 5: (Re)compute Buckets and Average Score
from pyspark.sql.functions import floor
df = df.withColumn("length_bucket", floor(df["review_length_chars"] / 50) * 50)
# Average review score per length bucket (will range from -1 to 1)
df.groupBy("length_bucket").avg("review_score").orderBy("length_bucket").show()

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1749504319068_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+------------+
|review_text                                                                                                                                                                                                                                                                                                                                                                                                                                      |review_length_chars|review_score|
+-------------------------------------------------------------

In [3]:
# Visual 1: Review Score Plot
import pandas as pd
import numpy
import matplotlib.pyplot as plt
import boto3

# Step 1: Group and collect
bucketed_df = df.groupBy("length_bucket") \
    .avg("review_score") \
    .orderBy("length_bucket") \
    .collect()

# Step 2: Convert to pandas
bucketed_df = pd.DataFrame(
    [(row['length_bucket'], row['avg(review_score)']) for row in bucketed_df],
    columns=["length_bucket", "avg_review_score"]
)

# Step 3: Plot
plt.figure(figsize=(12, 6))
plt.plot(bucketed_df["length_bucket"], bucketed_df["avg_review_score"], marker='o')
plt.title("Average Review Score by Review Length")
plt.xlabel("Review Length (bucketed by 50 characters)")
plt.ylabel("Average Review Score")
plt.grid(True)
plt.tight_layout()

# Step 4: Save plot locally
local_path = "/tmp/review_score_plot.png"
plt.savefig(local_path)

# Step 5: Upload to S3
s3 = boto3.client('s3')
bucket_name = "corey-final-project-bucket"
s3_path = "review_score_plot.png"

s3.upload_file(local_path, bucket_name, s3_path)
print(f"Plot saved and uploaded to s3://{bucket_name}/{s3_path}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Plot saved and uploaded to s3://corey-final-project-bucket/review_score_plot.png

In [4]:
# Visual 2: Review Length Plot
# Count how many reviews fall in each bucket
length_distribution = df.groupBy("length_bucket") \
    .count() \
    .orderBy("length_bucket") \
    .collect()

# Convert to pandas
length_dist_df = pd.DataFrame(
    [(row['length_bucket'], row['count']) for row in length_distribution],
    columns=["length_bucket", "review_count"]
)

# Plot as bar chart
plt.figure(figsize=(12, 6))
plt.bar(length_dist_df["length_bucket"], length_dist_df["review_count"], width=45, color='skyblue')
plt.title("Distribution of Review Lengths")
plt.xlabel("Review Length (bucketed by 50 characters)")
plt.ylabel("Number of Reviews")
plt.grid(True)
plt.tight_layout()

# Save and upload to S3
dist_path = "/tmp/review_length_distribution.png"
plt.savefig(dist_path)
s3.upload_file(dist_path, bucket_name, "review_length_distribution.png")
print(f"Distribution chart uploaded to s3://{bucket_name}/review_length_distribution.png")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Distribution chart uploaded to s3://corey-final-project-bucket/review_length_distribution.png

In [ ]:
# Visual 3: Weighted Review Score Plot
# Merge review count and average score by bucket
merged_df = pd.merge(bucketed_df, length_dist_df, on="length_bucket")

# Calculate weighted score for each bucket (this shows volume impact)
merged_df["weighted_score"] = merged_df["avg_review_score"] * merged_df["review_count"]

# Plot weighted average trend
plt.figure(figsize=(12, 6))
plt.plot(merged_df["length_bucket"], merged_df["weighted_score"], marker='o', color='green')
plt.title("Weighted Review Score by Review Length")
plt.xlabel("Review Length (bucketed by 50 characters)")
plt.ylabel("Weighted Score (Avg Score × Count)")
plt.grid(True)
plt.tight_layout()

# Save and upload to S3
weighted_path = "/tmp/weighted_review_score_plot.png"
plt.savefig(weighted_path)
s3.upload_file(weighted_path, bucket_name, "weighted_review_score_plot.png")
print(f"Weighted plot uploaded to s3://{bucket_name}/weighted_review_score_plot.png")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Weighted plot uploaded to s3://corey-final-project-bucket/weighted_review_score_plot.png

In [ ]:
# Classification Model
from pyspark.sql.functions import col
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Step 1: Load the cleaned dataset
df = spark.read.parquet("s3://corey-final-project-bucket/cleaned_steam_reviews/")

# Step 2: Cast review_score and review_votes to ints (again, just wanna be sure)
df = df.withColumn("label", ((col("review_score") + 1) / 2).cast("integer"))
df = df.withColumn("review_votes", col("review_votes").cast("int"))

# Step 3: Tokenize and clean text
tokenizer = Tokenizer(inputCol="review_text", outputCol="tokens")
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")

# Step 4: Text features via CountVectorizer
from pyspark.ml.feature import CountVectorizer
cv = CountVectorizer(inputCol="filtered", outputCol="rawFeatures", vocabSize=10000)
idf = IDF(inputCol="rawFeatures", outputCol="textFeatures")

# Step 5: Add numeric feature (review_votes)
assembler = VectorAssembler(inputCols=["textFeatures", "review_votes"], outputCol="features")

# Step 6: Classification model
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Step 7: Pipeline
pipeline = Pipeline(stages=[tokenizer, remover, cv, idf, assembler, lr])

# Step 8: Train/test split using 80/20 split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Step 9: Fit model
model = pipeline.fit(train_df)

# Step 10: Predict
predictions = model.transform(test_df)

# Step 11: Evaluate
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
roc_auc = evaluator.evaluate(predictions)
print(f"ROC AUC: {roc_auc}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

ROC AUC: 0.8911930700630295

In [7]:
# Inspecting most predictive words, so what words contributed most to pos or neg predictions using the log reg coefficients
# Step 1: Get the trained stages
cv_model = model.stages[2] # CountVectorizerModel
lr_model = model.stages[-1] # LogisticRegressionModel

# Step 2: Get vocabulary and coefficients
vocab = cv_model.vocabulary # List of words
coefficients = lr_model.coefficients.toArray()[:len(vocab)] # Text feature coefficients only

# Step 3: Zip words with coefficients
word_weights = list(zip(vocab, coefficients))

# Step 4: Sort by impact
top_positive = sorted(word_weights, key=lambda x: x[1], reverse=True)[:20]
top_negative = sorted(word_weights, key=lambda x: x[1])[:20]

# Step 5: Print
print("These are the top 20 words predicting positive reviews:")
for word, weight in top_positive:
    print(f"{word}: {weight:.4f}")

print("\nThese are the top 20 words predicting negative reviews:")
for word, weight in top_negative:
    print(f"{word}: {weight:.4f}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

These are the top 20 words predicting positive reviews:
10/10: 0.7175
best: 0.5390
addicting!: 0.4887
11/10: 0.4740
9/10: 0.4669
9.5/10: 0.4617
8/10: 0.4560
8.5/10: 0.4492
great: 0.4282
7.5/10: 0.4164
awesome: 0.3934
8/10.: 0.3897
10/10.: 0.3890
7/10.: 0.3741
100/10: 0.3696
underrated: 0.3662
amazing: 0.3656
12/10: 0.3619
must-play: 0.3581
addicting.: 0.3569

These are the top 20 words predicting negative reviews:
4/10: -0.3957
3/10: -0.3884
worst: -0.3633
boring: -0.3532
waste: -0.3392
boring.: -0.3283
2/10: -0.3027
refund.: -0.3021
unplayable.: -0.3013
refund: -0.2978
garbage.: -0.2882
disappointment.: -0.2852
unplayable: -0.2832
avoid.: -0.2733
1/10: -0.2714
sucks: -0.2568
terrible.: -0.2547
ruined: -0.2534
uninstall: -0.2531
0/10: -0.2511